Librerias

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# 1. Carregar les dades

In [2]:
# 1. Carregar les dades reals directament
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test.csv')

print(f"Éxito al cargar datos reales.")
print(f"Dimensiones de entrenamiento: {train_df.shape}")  # Debería mostrar (8693, 14)
print(f"Dimensiones de testeo: {test_df.shape}")         # Debería mostrar (4277, 13)

Éxito al cargar datos reales.
Dimensiones de entrenamiento: (8693, 14)
Dimensiones de testeo: (4277, 13)


# 2. Enginyeria de característiques (Feature Engineering)

In [3]:
def preprocess_features(df):
    df = df.copy()
    # Despesa total
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

    df[spend_cols] = df[spend_cols].fillna(0)
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    
    # Extreure grup del PassengerId
    df['Group'] = df['PassengerId'].apply(lambda x: x.split('_')[0] if pd.notnull(x) else '0000')
    
    # Extreure detalls de la cabina
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Cabin_Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Cabin_Side'] = df['Cabin'].apply(lambda x: x.split('/')[-1])
    return df

train_processed = preprocess_features(train_df)
test_processed = preprocess_features(test_df)

# Definir columnes numèriques i categòriques
num_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend']
cat_features = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side']

# Pipelines de preprocessament
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

# Separar característiques i target
X = train_processed[num_features + cat_features]
y = train_processed['Transported'].astype(int).values
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Aplicar transformacions
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
print(f"Forma de les dades d'entrenament transformades: {X_train_trans.shape}")

Forma de les dades d'entrenament transformades: (6954, 29)


# 3. Arquitectura de la Xarxa Neuronal (MLP)

In [4]:
# 3. Arquitectura de la Xarxa Neuronal (MLP)
model = keras.Sequential([
    keras.layers.Input(shape=(X_train_trans.shape[1],)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Early stopping per prevenir l'overfitting
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Entrenament del model

In [5]:

history = model.fit(
    X_train_trans, y_train,
    validation_data=(X_val_trans, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)
# Avaluació
val_preds = (model.predict(X_val_trans) > 0.5).astype(int)
acc = accuracy_score(y_val, val_preds)
print(f"\nAccuracy Final de Validació: {acc:.4f}")
print("\nInforme de Classificació:")
print(classification_report(y_val, val_preds))

Epoch 1/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7006 - loss: 0.5932 - val_accuracy: 0.7637 - val_loss: 0.4748
Epoch 2/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7679 - loss: 0.4815 - val_accuracy: 0.7694 - val_loss: 0.4384
Epoch 3/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7747 - loss: 0.4655 - val_accuracy: 0.7798 - val_loss: 0.4261
Epoch 4/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7826 - loss: 0.4537 - val_accuracy: 0.7798 - val_loss: 0.4216
Epoch 5/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7849 - loss: 0.4427 - val_accuracy: 0.7855 - val_loss: 0.4210
Epoch 6/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7915 - loss: 0.4407 - val_accuracy: 0.7763 - val_loss: 0.4207
Epoch 7/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7900 - loss: 0.4349 - val_accuracy: 0.7792 - val_loss: 0.4167
Epoch 8/50
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7942 - loss: 0.4335 - val_accuracy: 0.

# 4. Generar el archivo de envío para Kaggle

In [8]:
print("Preparando los datos de testeo para predicción...")

# 1. Aplicar exactamente el mismo preprocesamiento a los datos de testeo reales
# (Usamos las variables 'test_processed' y 'preprocessor' que ya tienes creadas en tu memoria)
X_test_real = test_processed[num_features + cat_features]
X_test_trans = preprocessor.transform(X_test_real)

# 2. Realizar predicciones con la red neuronal entrenada
# (Utilizamos el objeto 'model' que acabas de entrenar y está en memoria)
print("Realizando predicciones con el modelo...")
predictions_proba = model.predict(X_test_trans)

# 3. Convertir las probabilidades a valores booleanos (True/False) requeridos por Kaggle
# Si la probabilidad es igual o mayor a 0.5 es True, de lo contrario False
predictions_bool = (predictions_proba >= 0.5).astype(bool).flatten()

# 4. Crear el DataFrame de entrega usando el PassengerId del test_df real
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': predictions_bool
})

# 5. Guardar el archivo final listo para subir a Kaggle
output_filename = 'submission.csv'
submission_df.to_csv(output_filename, index=False)

print(f"\n¡Proceso completado con éxito! 🎉")
print(f"Tu archivo de entrega ha sido guardado como: '{output_filename}'")
print(f"Tiene un total de {len(submission_df)} filas.")

# Mostrar una vista previa de las primeras filas para asegurarnos de que todo está en orden
print("\nVista previa del archivo generado:")
print(submission_df.head())

Preparando los datos de testeo para predicción...
Realizando predicciones con el modelo...
134/134 ━━━━━━━━━━━━━━━━━━━━ 0s 664us/step

¡Proceso completado con éxito! 🎉
Tu archivo de entrega ha sido guardado como: 'submission.csv'
Tiene un total de 4277 filas.

Vista previa del archivo generado:
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
